# **Feature Engineering**

This notebook creates new features from the cleaned e-commerce dataset to support deeper analysis and modeling.  
Features are grouped by domain: Time, Financial, Discount, Customer, Product, Logistics, and Risk/Behavior.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
# Load the cleaned dataset
ecommerce_df = pd.read_csv(
    'dataset/cleaned/ecommerce_cleaned.csv',
    parse_dates=['order_date']
)

In [3]:
ecommerce_df.sample(5)

,order_date,order_year,order_month,is_weekend,customer_name,gender,age,customer_segment,country,order_status,category,sub_category,unit_price_usd,quantity,discount_percent,net_revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
849373,2024-12-30 14:51:43.860900,2024,12,No,Sarah Zimmerman,Female,68,Regular,France,Completed,Electronics,Smartphones,403.07,1,20,322.46,41.48,12.86,Apple Pay,Next Day,2.39,7,France,2,84.60,No,51.20,9,Yes,65.90,Desktop,Facebook,Direct
434533,2025-06-06 01:00:42.309671,2025,6,No,Brianna Clay,Female,24,Regular,Spain,Pending,Electronics,Smartphones,458.38,5,0,"2,291.90",970.10,42.33,Apple Pay,Express,12.20,3,Spain,3,79.00,Yes,20.60,17,No,9.90,Desktop,Email,Email
845072,2025-10-01 21:45:03.501191,2025,10,No,Johnathan Smith,Male,48,Regular,Netherlands,Completed,Sports,Gym Equipment,15.75,3,5,44.89,12.25,27.29,Apple Pay,Economy,11.82,4,Netherlands,1,26.20,No,10.20,6,No,31.70,Tablet,Organic,Social
958672,2024-10-04 10:44:59.773030,2024,10,No,Shawn Taylor,Male,45,VIP,Italy,Returned,Sports,Gym Equipment,76.98,1,10,69.28,29.81,43.03,Credit Card,Express,18.18,10,Italy,5,23.80,Yes,31.20,10,No,40.80,Tablet,Google Ads,Search
629831,2024-02-15 04:09:33.128778,2024,2,No,Nancy Williams,Female,19,VIP,Australia,Completed,Health,Supplements,125.94,1,0,125.94,59.33,47.11,Bank Transfer,Express,1.83,12,Australia,5,50.20,Yes,34.30,7,No,86.50,Mobile,Affiliate,Direct


## 1. Time Features

In [4]:
# Day of week (Mon, Tue, ...)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_month') + 1,
    'order_day_of_week',
    ecommerce_df['order_date'].dt.strftime('%a')
)

# Part of day
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_day_of_week') + 1,
    'part_day',
    pd.cut(
        ecommerce_df['order_date'].dt.hour,
        bins=[0, 6, 12, 18, 24],
        labels=['Night', 'Morning', 'Afternoon', 'Evening'],
        right=False
    )
)

## 2. Financial / Revenue Features

In [5]:
# Gross revenue before discount
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'gross_revenue_usd',
    ecommerce_df.eval('quantity * unit_price_usd')
)

# Actual discount amount
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('gross_revenue_usd') + 1,
    'discount_amount_usd',
    ecommerce_df.eval('gross_revenue_usd * discount_percent / 100')
)

# Shipping cost as % of revenue
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_cost_usd') + 1,
    'shipping_cost_percent',
    ecommerce_df.eval('shipping_cost_usd / net_revenue_usd * 100')
)

## 3. Discount Features

In [6]:
# Discount tier (adjusted to actual data range 0-25%)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_percent') + 1,
    'discount_tier',
    pd.cut(
        ecommerce_df['discount_percent'],
        bins=[-1, 0, 10, 20, 25],
        labels=['No Discount', 'Low Discount', 'Medium Discount', 'High Discount']
    )
)

# Binary flag
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_tier') + 1,
    'is_discounted',
    np.where(
        ecommerce_df.eval('discount_percent > 0'),
        "Yes",
        "No"
    )
)

## 4. Customer Features

In [7]:
# Age group
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('age') + 1,
    'age_group',
    pd.cut(
        ecommerce_df['age'],
        bins=[0, 24, 34, 44, 54, 64, 100],
        labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
    )
)

# Loyalty tier
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('customer_loyalty_score') + 1,
    'loyalty_tier',
    pd.cut(
        ecommerce_df['customer_loyalty_score'],
        bins=[-1, 30, 70, 100],
        labels=['Low', 'Medium', 'High']
    )
)

## 5. Product & Order Features

In [8]:
# Price tier
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('unit_price_usd') + 1,
    'price_tier',
    pd.cut(
        ecommerce_df['unit_price_usd'],
        bins=[0, 50, 100, 200, np.inf],
        labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
    )
)

# Quantity / basket size
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'quantity_segment',
    pd.cut(
        ecommerce_df['quantity'],
        bins=[0, 1, 3, 5],
        labels=['Single', 'Small Basket', 'Bulk']
    )
)

# Order value segment
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('net_revenue_usd') + 1,
    'order_value_segment',
    pd.cut(
        ecommerce_df['net_revenue_usd'],
        bins=[0, 100, 300, 600, np.inf],
        labels=['Low', 'Medium', 'High', 'Very High']
    )
)

## 6. Logistics Features

In [9]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('delivery_days') + 1,
    'delivery_speed',
    pd.cut(
        ecommerce_df['delivery_days'],
        bins=[0, 3, 7, 15],
        labels=['Fast', 'Standard', 'Slow']
    )
)
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_country') + 1,
    'shipment_type',
    np.where(
        ecommerce_df['country'] == ecommerce_df['shipping_country'],
        "Domestic",
        "International"
    )
)

## 7. Behavior & Risk Features

In [10]:
# Engagement level
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('session_duration_min') + 1,
    'engagement_level',
    pd.cut(
        ecommerce_df['session_duration_min'],
        bins=[-1, 15, 40, 100],
        labels=['Low', 'Medium', 'High']
    )
)

# Fraud risk level
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('fraud_risk_score') + 1,
    'fraud_risk_level',
    pd.cut(
        ecommerce_df['fraud_risk_score'],
        bins=[-1, 30, 70, 100],
        labels=['Low', 'Medium', 'High']
    )
)

## 8. Final Check & Save

In [11]:
# Quick look at the engineered dataset
ecommerce_df.sample(10)

,order_date,order_year,order_month,order_day_of_week,part_day,is_weekend,customer_name,gender,age,age_group,customer_segment,country,order_status,category,sub_category,unit_price_usd,price_tier,quantity,quantity_segment,gross_revenue_usd,discount_amount_usd,discount_percent,discount_tier,is_discounted,net_revenue_usd,order_value_segment,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,shipping_cost_percent,delivery_days,delivery_speed,shipping_country,shipment_type,rating,customer_loyalty_score,loyalty_tier,coupon_used,session_duration_min,engagement_level,pages_visited,abandoned_cart_before,fraud_risk_score,fraud_risk_level,device_type,campaign_source,traffic_source
544746,2025-09-13 04:13:21.603143,2025,9,Sat,Night,Yes,Kristin Warren,Female,29,25-34,Regular,United States,Completed,Electronics,Smartphones,204.98,Luxury,5,Bulk,"1,024.90",102.49,10,Low Discount,Yes,922.41,Very High,256.21,27.78,Apple Pay,Express,16.80,1.82,4,Standard,United States,Domestic,2,39.70,Medium,No,27.50,Medium,11,No,80.20,High,Tablet,Instagram,Email
929581,2025-07-11 11:11:32.001621,2025,7,Fri,Morning,No,Matthew Lopez,Male,18,18-24,Regular,United States,Completed,Home,Furniture,99.88,Mid-Range,3,Small Basket,299.64,44.95,15,Medium Discount,Yes,254.69,Medium,98.96,38.86,Bank Transfer,Express,22.74,8.93,4,Standard,United States,Domestic,1,97.80,High,Yes,14.30,Low,5,Yes,79.20,High,Tablet,Instagram,Search
329661,2025-10-08 07:25:05.761435,2025,10,Wed,Morning,No,Michelle Smith,Female,59,55-64,Premium,Italy,Completed,Electronics,Tablets,398.97,Luxury,1,Single,398.97,19.95,5,Low Discount,Yes,379.02,High,171.22,45.17,Debit Card,Express,8.41,2.22,4,Standard,Italy,Domestic,1,76.30,High,No,9.30,Low,8,Yes,18.10,Low,Desktop,Email,Social
567063,2025-03-03 06:28:39.364129,2025,3,Mon,Morning,No,Stacy Hernandez,Female,39,35-44,VIP,Netherlands,Completed,Clothing,Shoes,22.98,Budget,3,Small Basket,68.94,0.00,0,No Discount,No,68.94,Low,26.52,38.47,Debit Card,Next Day,17.15,24.88,4,Standard,Netherlands,Domestic,3,99.80,High,No,26.80,Medium,13,Yes,98.30,High,Desktop,Facebook,Social
304265,2025-09-16 16:02:43.285474,2025,9,Tue,Afternoon,No,Ian Lawson,Male,56,55-64,Premium,Australia,Completed,Home,Bedding,159.81,Premium,2,Small Basket,319.62,0.00,0,No Discount,No,319.62,High,111.12,34.77,Apple Pay,Standard,12.84,4.02,13,Slow,Australia,Domestic,4,71.80,High,Yes,30.90,Medium,1,Yes,76.50,High,Desktop,Affiliate,Direct
367099,2025-05-12 05:27:44.845630,2025,5,Mon,Night,No,Mary Shelton,Female,25,25-34,Regular,Germany,Pending,Health,Personal Care,79.49,Mid-Range,3,Small Basket,238.47,47.69,20,Medium Discount,Yes,190.78,Medium,45.64,23.92,Debit Card,Express,1.90,1.00,13,Slow,Germany,Domestic,3,26.00,Low,Yes,39.00,Medium,13,No,56.40,Medium,Tablet,Organic,Direct
1809,2025-05-16 21:18:32.598495,2025,5,Fri,Evening,No,Emily Quinn,Female,33,25-34,Premium,Italy,Completed,Health,Supplements,181.98,Premium,5,Bulk,909.90,45.49,5,Low Discount,Yes,864.41,Very High,231.91,26.83,Debit Card,Standard,8.55,0.99,6,Standard,Italy,Domestic,3,6.50,Low,No,33.00,Medium,11,Yes,92.80,High,Mobile,Instagram,Social
96753,2024-08-07 15:43:43.823845,2024,8,Wed,Afternoon,No,Matthew Jimenez,Male,54,45-54,VIP,Italy,Completed,Electronics,Tablets,348.93,Luxury,2,Small Basket,697.86,0.00,0,No Discount,No,697.86,Very High,239.76,34.36,Apple Pay,Next Day,3.63,0.52,12,Slow,Italy,Domestic,3,84.30,High,No,49.30,High,9,Yes,14.80,Low,Mobile,Affiliate,Direct
236833,2025-10-16 12:59:27.302626,2025,10,Thu,Afternoon,No,Jamie Brown,Female,42,35-44,Regular,France,Completed,Sports,Gym Equipment,21.35,Budget,4,Bulk,85.40,21.35,25,High Discount,Yes,64.05,Low,6.73,10.51,Bank Transfer,Economy,14.37,22.44,6,Standard,France,Domestic,4,31.40,Medium,Yes,21.70,Medium,15,No,92.00,High,Tablet,Organic,Referral
945059,2025-03-10 20:18:14.167470,2025,3,Mon,Evening,No,Kathy Allison,Female,47,45-54,Premium,Italy,Completed,Electronics,Accessories,82.46,Mid-Range,5,Bulk,412.30,82.46,20,Medium Discount,Yes,329.84,High,

In [12]:
# Save the processed dataset
ecommerce_df.to_csv("dataset/processed/ecommerce_processed.csv", index=False)